In [1]:
import subprocess
import os
import pandas as pd
import numpy as np
from core.reader import read_ansys_csv, load_experimental_data

ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

def run_ansys_simulation(params):
    with open('chab_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")

    input_file = "chab.mac"
    output_file = "ansys.out"
    
    cmd = [
        ANSYS_EXE_PATH, 
        "-b",
        "-j", "opt_run",
        "-dir", WORKING_DIR, 
        "-i", input_file, 
        "-o", output_file
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("Ошибка ANSYS:", e)
        return None

    try:
        df_res = read_ansys_csv("chab.csv")
        return df_res
    except Exception as e:
        print(f"Ошибка чтения CSV: {e}")
        return None

In [ ]:
real_experiment_data_folder = "."
df_exp = load_experimental_data(real_experiment_data_folder)

zero_row = pd.DataFrame(0.0, columns=df_exp.columns, index=[0])
zero_row['Time'] = 1.0
df_exp = pd.concat([zero_row, df_exp]).reset_index(drop=True)

def objective_function(params):
    """
    Считает ошибку между экспериментом и моделью Шабоша.
    params: [sig_y, c1, g1, c2, g2, c3, g3]
    """
    print(f"Simulating: {params}")
    
    df_ansys = run_ansys_simulation(params)
    
    if df_ansys is None or len(df_ansys) != len(df_exp):
        return 1e9

    mse_zz = np.mean((df_ansys['S_ZZ'] - df_exp['S_ZZ'])**2)
    mse_tt = np.mean((df_ansys['S_TT'] - df_exp['S_TT'])**2)
    mse_tz = np.mean((df_ansys['S_TZ'] - df_exp['S_TZ'])**2)
    
    total_error = mse_zz + mse_tt + mse_tz
    print(f"Error: {total_error:.2f}")
    return total_error

In [3]:
import psutil

def kill_ansys_processes():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys_processes()

In [5]:
from scipy.optimize import differential_evolution

# Границы поиска для каждого параметра
# [sig_y, c1, g1, c2, g2, c3, g3]
bounds = [
    (200, 400),      # Sig_Y
    (1e4, 5e5),      # C1 (Жесткая кинематика)
    (100, 5000),     # gamma1 (Быстрое насыщение)
    (1e3, 5e4),      # C2
    (10, 500),       # gamma2
    (100, 1e4),      # C3
    (0, 100)         # gamma3 (Линейная часть)
]

result = differential_evolution(
    objective_function, 
    bounds, 
    strategy='best1bin', 
    maxiter=15,      # Количество поколений (увеличьте до 20-50 для точности)
    popsize=10,       # Размер популяции (увеличьте до 10-15)
    disp=True,
    polish=True,
    workers=1
)

print("Оптимальные параметры найдены:")
print(result.x)

Simulating: [3.88663649e+02 4.01843361e+05 6.57757262e+02 3.26300785e+04
 3.90637721e+02 7.96100361e+03 8.40138252e+01]
Error: 2.14
Simulating: [2.81167631e+02 1.76694001e+05 1.35888050e+03 8.95783753e+03
 2.26707269e+02 5.98245176e+03 7.91819174e+01]
Error: 0.05
Simulating: [3.49946878e+02 3.69015729e+05 7.33700615e+02 1.47104732e+04
 8.11626247e+01 7.84155932e+03 9.36135382e+00]
Error: 1.68
Simulating: [3.16902982e+02 6.74954210e+04 1.77463002e+02 3.16862354e+04
 2.69951370e+02 6.35609962e+03 1.93316391e+01]
Error: 0.56
Simulating: [2.03446331e+02 4.76017250e+05 3.62371453e+03 1.77838302e+04
 3.67227989e+02 9.74905922e+03 1.15706220e+01]
Error: 0.08
Simulating: [2.62700806e+02 2.37756984e+05 4.38986047e+02 4.44347736e+04
 2.80784091e+02 6.69210035e+03 1.91897380e+00]
Error: 1.42
Simulating: [3.72429003e+02 1.07429104e+05 3.31378222e+03 3.74135550e+04
 3.29230105e+02 9.11331978e+03 8.86747267e+01]
Error: 0.08
Simulating: [3.32950950e+02 3.47567222e+05 3.11940435e+03 4.15479675e+04
 2.

In [ ]:
# from scipy.optimize import minimize

# # Start Point = [sig_y, c1, g1, c2, g2, c3, g3]
# x0 = [
#     243.796231,    # Sig_Y
#     383442.628, # C1
#     1932.38252,   # gamma1
#     7087.97318,  # C2
#     400.643728,     # gamma2
#     2846.64799,   # C3
#     1.22110748      # gamma3
# ]
# print(f"Запуск локальной оптимизации (Nelder-Mead) с начальной точки:\n{x0}")

# bounds = [
#     (200, 400),           # Sig_Y
#     (1e4, 5e5),           # C1
#     (100, 5000),          # gamma1
#     (1e3, 5e4),           # C2
#     (10, 500),            # gamma2
#     (100, 1e4),           # C3
#     (0, 100)             # gamma3
# ]

# res = minimize(
#     objective_function, 
#     x0, 
#     method='Nelder-Mead',
#     bounds=bounds,
#     options={
#         'maxiter': 200,    # Количество запусков ANSYS
#         'disp': True,
#         'xatol': 1.0,     # Точность поиска
#         'fatol': 100.0    # Точность по функции ошибки
#     }
# )

# print("\nОптимизация завершена!")
# print("Лучшие параметры:", res.x)
# print("Лучшая ошибка:", res.fun)

Запуск локальной оптимизации (Nelder-Mead) с начальной точки:
[243.796231, 383442.628, 1932.38252, 7087.97318, 400.643728, 2846.64799, 1.22110748]
Simulating: [2.43796231e+02 3.83442628e+05 1.93238252e+03 7.08797318e+03
 4.00643728e+02 2.84664799e+03 1.22110748e+00]
Error: 0.05
Simulating: [2.55986043e+02 3.83442628e+05 1.93238252e+03 7.08797318e+03
 4.00643728e+02 2.84664799e+03 1.22110748e+00]
Error: 0.05
Simulating: [2.43796231e+02 4.02614759e+05 1.93238252e+03 7.08797318e+03
 4.00643728e+02 2.84664799e+03 1.22110748e+00]
Error: 0.05
Simulating: [2.43796231e+02 3.83442628e+05 2.02900165e+03 7.08797318e+03
 4.00643728e+02 2.84664799e+03 1.22110748e+00]
Error: 0.05
Simulating: [2.43796231e+02 3.83442628e+05 1.93238252e+03 7.44237184e+03
 4.00643728e+02 2.84664799e+03 1.22110748e+00]
Error: 0.05
Simulating: [2.43796231e+02 3.83442628e+05 1.93238252e+03 7.08797318e+03
 4.20675914e+02 2.84664799e+03 1.22110748e+00]
Error: 0.05
Simulating: [2.43796231e+02 3.83442628e+05 1.93238252e+03 7.0

C:\Users\Arina\AppData\Local\Temp\ipykernel_1820\335314971.py:25: RuntimeWarning: Maximum number of iterations has been exceeded.
  res = minimize(


In [ ]:
# x0 = [
#     209.605994,   # параметр 1
#     463629.692,  # параметр 2
#     1823.87641,  # параметр 3
#     1163.27023,  # параметр 4
#     498.512508,  # параметр 5
#     2784.66372,  # параметр 6
#     2.13799691   # параметр 7
# ]



# initial_error = objective_function(x0)
# print(f"Ошибка на старте: {initial_error}")

Simulating: [209.605994, 463629.692, 1823.87641, 1163.27023, 498.512508, 2784.66372, 2.13799691]
Error: 0.05
Ошибка на старте: 0.04568123749327316
